In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

In [7]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [9]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [18]:
categorical  = ['lead_source', 'industry', 'employment_status', 'location']
numerical  = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [19]:
df[categorical ] = df[categorical ].fillna('NA')
df[numerical ] = df[numerical ].fillna(0.0)

In [20]:
df[categorical ].head()

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NA
4,referral,manufacturing,student,north_america


In [22]:
df.isnull().sum()


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [14]:
df['industry'].mode()[0]

'technology'

In [23]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [24]:
df[numerical].corr() 

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [25]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [26]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [27]:
y_train = df_train.converted.values
y_val = df_val.converted.values

In [28]:
del df_train['converted']
del df_val['converted']

In [29]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [30]:
mi = df_train[categorical].apply(mutual_info_converted_score)
mi.round(2).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [31]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
 
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
 
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)
 
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
 
    y_pred = model.predict_proba(X_val)[:, 1]
    decision = (y_pred >= 0.5)
    return (decision == y_val).mean()

In [32]:
features = categorical + numerical


In [33]:
features

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [34]:
base_accuracy = train_and_score(features)


In [35]:
base_accuracy

np.float64(0.645)

In [36]:
print(round(base_accuracy, 2))

0.64


In [37]:
for f in features:
    subset = [x for x in features if x != f]
    diff = base_accuracy - train_and_score(subset)
    print(f, diff)

lead_source 0.0030000000000000027
industry 0.0
employment_status 0.0
location 0.0
annual_income -0.07899999999999996
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004
lead_score 0.0010000000000000009


In [38]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
